# Visualizador Gradio dos Parquets

Abre um app Gradio read-only para navegar pelos Parquets `textos_parlamentares/v1`. No Colab, usa os Parquets completos no Google Drive. Fora do Colab, usa os Parquets de samples locais quando existirem.

In [ ]:
try:
    from google.colab import drive
except ImportError:
    IN_COLAB = False
    print('Ambiente local detectado; Google Drive nao sera montado.')
else:
    IN_COLAB = True
    drive.mount('/content/drive')

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/pedblan/falando_nela.git'
REPO_DIR = Path('/content/falando_nela') if IN_COLAB else Path.cwd().resolve()
REPO_REF = ''  # opcional: branch, tag ou commit

if IN_COLAB:
    if not REPO_DIR.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=False)
    if REPO_REF:
        subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', REPO_REF], check=True)
else:
    for candidate in [REPO_DIR, *REPO_DIR.parents]:
        if (candidate / 'requirements.txt').exists() and (candidate / 'notebooks').exists():
            REPO_DIR = candidate
            break

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print('Repositorio:', REPO_DIR)

In [ ]:
import subprocess
import sys

PACKAGES = [
    'altair>=5,<6',
    'duckdb>=1,<2',
    'gradio>=4,<6',
    'pandas>=2,<3',
    'pyarrow>=15,<23',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PACKAGES], check=True)

In [ ]:
from pathlib import Path
import os

DATA_ROOT = Path('/content/drive/MyDrive/falando_nela/data')
COLAB_PARQUET_ROOT = Path('/content/drive/MyDrive/falando_nela/data/processed/textos_parlamentares/v1/parquet')
SAMPLES_PARQUET_ROOT = REPO_DIR / 'data' / 'samples' / 'textos_parlamentares' / 'v1' / 'parquet'

DATA_PROFILE = 'colab-drive' if IN_COLAB else 'samples-local'
PARQUET_ROOT = COLAB_PARQUET_ROOT if IN_COLAB else SAMPLES_PARQUET_ROOT
os.environ['FALANDO_NELA_DATA_ROOT'] = str(DATA_ROOT)

print('Perfil:', DATA_PROFILE)
print('PARQUET_ROOT:', PARQUET_ROOT)

In [ ]:
import gradio as gr

from processamento.visualizador_parquets import build_gradio_app, list_parquet_files

parquet_files = list_parquet_files(PARQUET_ROOT)
print(f'{len(parquet_files)} Parquet(s) encontrado(s).')
for name in parquet_files:
    print('-', name)

## Iniciar app

O app abaixo le somente os Parquets do diretorio configurado. Para ver dados recem-atualizados, gere novamente os Parquets no fluxo apropriado e relance ou atualize a lista do app.

In [ ]:
app = build_gradio_app(PARQUET_ROOT)
if IN_COLAB:
    app.launch(share=True)
else:
    app.launch(share=False)